<a href="https://colab.research.google.com/github/Aleppoo/Aleppoo/blob/main/CarPlateDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics
!pip install ultralytics pytesseract
!sudo apt install tesseract-ocr

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from google.colab.patches import cv2_imshow

In [ ]:
import os
import glob
import cv2
import numpy as np
import shutil
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO
import pytesseract
from IPython.display import display, Image
from google.colab import files
from google.colab.patches import cv2_imshow
import albumentations as A

In [ ]:
def load_image_and_label(image_path, label_path):
    try:
        image = cv2.imread(image_path)
        if image is None:
            raise ValueError(f"Could not read image: {image_path}")
        image_height, image_width, _ = image.shape

        if not os.path.exists(label_path):
            raise FileNotFoundError(f"Label file not found: {label_path}")

        with open(label_path, "r") as f:
            label_line = f.readline().strip()
        if not label_line:
            print(f"Warning: Empty label file: {label_path}")
            return image, None, None
        label_parts = label_line.split()
        if len(label_parts) != 5:
            raise ValueError(f"Invalid label format in {label_path}: {label_line}")
        class_id = int(label_parts[0])
        x_center = float(label_parts[1])
        y_center = float(label_parts[2])
        width = float(label_parts[3])
        height = float(label_parts[4])

        x_min = int((x_center - width / 2) * image_width)
        y_min = int((y_center - height / 2) * image_height)
        x_max = int((x_center + width / 2) * image_width)
        y_max = int((x_center + width / 2) * image_width)

        bounding_box = [x_min, y_min, x_max, y_max]
        return image, bounding_box, class_id

    except (ValueError, FileNotFoundError, IndexError) as e:
        print(f"Error loading image/label: {e}")
        return None, None, None

def preprocess_sharpen(image):
    kernel = np.array([[-1,-1,-1], [-1,9,-1], [-1,-1,-1]])
    sharpened_image = cv2.filter2D(image, -1, kernel)
    return sharpened_image

data_dir = "/content/drive/MyDrive/data"
subsets = ["train", "val"]

processed_data_dir = os.path.join(data_dir, "processed_images")
os.makedirs(processed_data_dir, exist_ok=True)

display_count = 0

for subset in subsets:
    print(f"--- Starting processing for subset: {subset} ---")
    processed_subset_dir = os.path.join(processed_data_dir, subset)
    processed_labels_dir = os.path.join(processed_subset_dir, "labels")
    processed_images_dir = os.path.join(processed_subset_dir, "images")
    os.makedirs(processed_images_dir, exist_ok=True)
    os.makedirs(processed_labels_dir, exist_ok=True)
    print(f"Processed images directory: {processed_images_dir}")
    print(f"Processed labels directory: {processed_labels_dir}")

    image_dir = os.path.join(data_dir, subset, "images")
    label_dir = os.path.join(data_dir, subset, "labels")
    print(f"Image directory: {image_dir}")
    print(f"Label directory: {label_dir}")

    if not os.path.exists(image_dir):
        print(f"ERROR: Image directory NOT FOUND: {image_dir}")
    if not os.path.exists(label_dir):
        print(f"ERROR: Label directory NOT FOUND: {label_dir}")

    if not os.path.exists(image_dir) or not os.path.exists(label_dir):
        print(f"Skipping subset {subset} due to missing directories.")
        continue

    image_files = glob.glob(os.path.join(image_dir, "*.jpg"))
    print(f"Found {len(image_files)} image files in {image_dir}")

    for image_path in image_files:
        image_name = os.path.basename(image_path)
        label_name = image_name.replace(".jpg", ".txt")
        label_path = os.path.join(label_dir, label_name)
        processed_label_path = os.path.join(processed_labels_dir, label_name)

        if not os.path.exists(label_path):
            print(f"Warning: No label file found for {image_name}")
            continue

        image, bounding_box, class_id = load_image_and_label(image_path, label_path)

        if image is not None:
            sharpened_image = preprocess_sharpen(image)
            output_filename = os.path.join(processed_images_dir, image_name)
            cv2.imwrite(output_filename, sharpened_image)
            shutil.copy2(label_path, processed_label_path)

            if display_count < 5:
                plt.figure(figsize=(10, 5))

                plt.subplot(1, 2, 1)
                plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
                plt.title("Original Image")
                plt.axis('off')

                plt.subplot(1, 2, 2)
                plt.imshow(cv2.cvtColor(sharpened_image, cv2.COLOR_BGR2RGB))
                plt.title("Sharpened Image")
                plt.axis('off')

                plt.show()
                display_count += 1

print("Finished processing all subsets.")

In [ ]:
data_yaml_path = os.path.join(data_dir, "data.yaml")
with open(data_yaml_path, "w") as f:
    train_images_path = Path(os.path.join(processed_data_dir, 'train', 'images')).resolve()
    train_labels_path = Path(os.path.join(processed_data_dir, 'train', 'labels')).resolve()
    val_images_path = Path(os.path.join(processed_data_dir, 'val','images')).resolve()
    val_labels_path = Path(os.path.join(processed_data_dir, 'val','labels')).resolve()

    f.write(f"path: {str(Path(processed_data_dir).resolve())}\n")
    f.write(f"train: train/images\n")
    f.write(f"val: val/images\n")
    f.write(f"nc: 1\n")
    f.write("names: ['car_plate']\n")

print(f"data.yaml contents written to: {data_yaml_path}")
print(f"Train images path in data.yaml: {train_images_path}")
print(f"Train labels path in data.yaml: {train_labels_path}")
print(f"Val images path in data.yaml: {val_images_path}")
print(f"Val labels path in data.yaml: {val_labels_path}")

model = YOLO('yolov8s.pt')

model.train(data=data_yaml_path, epochs=100, imgsz=640, project=data_dir)

metrics = model.val()

print(metrics)

In [ ]:
model_path = r"/content/drive/MyDrive/data/yolov8s.pt"
model = YOLO(model_path)

In [ ]:
uploaded = files.upload()
for fn in uploaded.keys():
    uploaded_image = fn
    image = cv2.imread(uploaded_image)

    results = model.predict(source=image, save=False)

    if results and len(results[0].boxes) > 0:
        for box in results[0].boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            cropped_plate = image[y1:y2, x1:x2]

            gray = cv2.cvtColor(cropped_plate, cv2.COLOR_BGR2GRAY)
            text = pytesseract.image_to_string(gray, config='--psm 8')
            print(f"Detected Plate Text: {text.strip()}")
            cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
            display(Image(data=cv2.imencode('.jpg', image)[1].tobytes()))
            break
    else:
      print("No car registration plate detected in the image.")
      display(Image(data=cv2.imencode('.jpg', image)[1].tobytes()))